[![Python](https://img.shields.io/badge/Python-3776AB?logo=python&logoColor=fff)](#) [![Google Colab](https://img.shields.io/badge/Google%20Colab-F9AB00?logo=googlecolab&logoColor=fff)](#) [![Pandas](https://img.shields.io/badge/Pandas-150458?logo=pandas&logoColor=fff)](#)

---
<h4>
<h2><center><b> Prova Técnica - Operações B2C com Foco em Dados <br>
<font color = "blue"> Agibank </center></b><br>

---

<center><font size = 4><b> Alexandre Galetti </center>

## **Introdução**

---

Você está atuando na área de operações B2C de uma instituição financeira. Diariamente recebemos bases com solicitações de clientes que precisam ser tratadas e transformadas em controles operacionais e análises.

## **Configurações iniciais**

---

In [2]:
# Criando a pasta compartilhada
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Bibliotecas utilizadas
import numpy as np        # funções matemáticas e operações com vetores e matrizes
import pandas as pd       # estruturas tabulares de organização e análise de dados
import matplotlib as plt  # geração e visualização de gráficos


## **Parte I - Entendimento do Problema**

---

In [4]:
# Análise preliminar do dataset

df = pd.read_csv('/content/drive/MyDrive/Agibank/Base dados Prova Técnica.csv')
display(df)

,id_solicitacao,data_solicitacao,cpf_cliente,produto,valor_solicitado,status,canal_entrada,cidade,analista_responsavel
0,1,2025-01-18,82919005427,Emprestimo,16658.0,Aprovado,Web,Recife,Carlos
1,2,2025-01-12,57532219711,Cartao,7094.0,Aprovado,Web,Rio de Janeiro,Carlos
2,3,2025/02/25,41885063984,Emprestimo,75561.0,Pendente,App,Porto Alegre,Julia
3,4,14/02/2025,16068880053,Cartao,13896.0,Aprovado,Web,Fortaleza,Lucas
4,5,25/01/2025,75199612656,Cartao,15978.0,Negado,App,Porto Alegre,Lucas
...,...,...,...,...,...,...,...,...,...
1015,974,19/01/2025,28092000962,Financiamento,64304.0,Negado,Agencia,Fortaleza,Fernanda
1016,308,2025-02-19,88326560670,Cartao,75151.0,Pendente,Web,Salvador,Carlos
1017,847,2025-01-13,16414194238,Emprestimo,9290.0,Aprovado,Web,Goiania,Ana
1018,548,08/02/2025,67587456574,Financiamento,NaN,Negado,Agencia,Recife,Carlos


**<u>Comentário**</u>: A análise preliminar do dataset indica que é um levantamento das solicitações de crédito da instituição financeira, separada nas categorias `cartão`, `empréstimo` e `financiamento`. Posso ver que as informações mais relevantes são: valor solicitado, status da proposta (aprovado/negado/pendente), canal de solicitação (presencial/online) e localização geográfica dos clientes.

In [5]:
df.describe() # medidas estatísticas das colunas com valores numéricos

,id_solicitacao,cpf_cliente,valor_solicitado
count,1020.000000,1.020000e+03,700.000000
mean,502.696078,5.461985e+10,40418.711429
std,288.939457,2.628377e+10,35459.656289
min,1.000000,1.010345e+10,500.000000
25%,252.750000,3.208057e+10,10938.250000
50%,503.500000,5.313792e+10,24328.500000
75%,754.250000,7.742556e+10,69045.250000
max,1000.000000,9.993611e+10,119846.000000


**<u>Comentário**</u>: Note que existe uma divergência; a contagem de id é de 1020 elementos mas a coluna `valor_solicitado` possui apenas 700. Há <u>dados faltantes</u> nesta última coluna, preciso corrigir primeiro.

In [6]:
df['valor_solicitado'].describe() # medidas estatísticas somente da coluna de interesse

,valor_solicitado
count,700.000000
mean,40418.711429
std,35459.656289
min,500.000000
25%,10938.250000
50%,24328.500000
75%,69045.250000
max,119846.000000


In [7]:
# Vou substituir a informação "NaN" por 0 para facilitar a leitura
# e corrigir os cálculos posteriores do dataset

df['valor_solicitado'] = df['valor_solicitado'].fillna(0)
display(df)

,id_solicitacao,data_solicitacao,cpf_cliente,produto,valor_solicitado,status,canal_entrada,cidade,analista_responsavel
0,1,2025-01-18,82919005427,Emprestimo,16658.0,Aprovado,Web,Recife,Carlos
1,2,2025-01-12,57532219711,Cartao,7094.0,Aprovado,Web,Rio de Janeiro,Carlos
2,3,2025/02/25,41885063984,Emprestimo,75561.0,Pendente,App,Porto Alegre,Julia
3,4,14/02/2025,16068880053,Cartao,13896.0,Aprovado,Web,Fortaleza,Lucas
4,5,25/01/2025,75199612656,Cartao,15978.0,Negado,App,Porto Alegre,Lucas
...,...,...,...,...,...,...,...,...,...
1015,974,19/01/2025,28092000962,Financiamento,64304.0,Negado,Agencia,Fortaleza,Fernanda
1016,308,2025-02-19,88326560670,Cartao,75151.0,Pendente,Web,Salvador,Carlos
1017,847,2025-01-13,16414194238,Emprestimo,9290.0,Aprovado,Web,Goiania,Ana
1018,548,08/02/2025,67587456574,Financiamento,0.0,Negado,Agencia,Recife,Carlos


In [8]:
df['valor_solicitado'].describe() # medidas estatísticas somente da coluna de interesse corrigida

,valor_solicitado
count,1020.000000
mean,27738.331373
std,34851.161242
min,0.000000
25%,0.000000
50%,11650.500000
75%,50604.000000
max,119846.000000


**<u>Comentário**</u>: O número de elementos (linhas) da coluna valor solicitado corresponde ao id e está de acordo com o número de linhas do dataset. Note que as medidas estatísticas mudaram e são, agora, mais confiáveis. O alerta que fazemos é que 1/4 ou 25% dos dados (255 elementos) não possuem a informação de valor solicitado; isso pode impactar a analise a seguir.

> Minha sugestão seria informar o gestor da área se é um problema tecnológico ou operacional e propor melhorias no processo para otimizar essa coleta de informações.

### **Respostas das perguntas da Parte I**

---

**•	Quais problemas de qualidade de dados você identifica.**


- Id fora de ordem (últimas solicitações);
- Data solicitação com formatos diferentes (não padronizados);
- Valor solicitado com formatação não contábil que dificulta a leitura (ideal xxx.xxx,xx) + dados não estruturados (NaN);
- A formatação de valores não está numa estrutura clara que permite identificar a ordem de grandeza, pode gerar confusão e erros se não prestar atenção.



---

**•	Quais validações você implementaria antes de utilizar esses dados.**


- Organização em ordem crescente do Id;
- Padronização do formato de data (dd/mm/aaaa);
- Padronização do formato contábil (financeiro R$) e substituição da informação NaN por “Não disponível” ou valor numérico (ideal é zero para não comprometer a análise estatística depois).
- Verificação se número de linhas do dataset é igual a contagem obtida com o método `pd.describe()` = basicamente isso indica se existem dados faltantes que requerem maior atenção.


---

**•	Como estruturaria um processo de tratamento diário dessa base.**


**0.	Elaboração da documentação**: geração de um arquivo `readme` contendo as instruções e passo a passo deste processo. O objetivo é documentar o contexto do problema, os objetivos que buscamos, resultados e, principalmente, as atualizações que poderão ocorrer com o tempo, seja do processo em si, quanto a forma no tratamento de dados.

**1.	Validação dos dados utilizando Python (biblioteca Pandas) e Excel**:
Por que utilizar ambos? Para ter uma dupla validação e visualizar os dados em ambientes diferentes permitem insights mais detalhados e com menos erros.
Verificar tamanho e estrutura da base. Verificar se os dados em cada coluna correspondem ao tipo correto (número = dados numéricos; CPF = número de CPF com 11 dígitos _ embora seja tratado como `string` pois não podemos fazer operações matemáticas com ele; valor solicitado = moeda etc.)

**2.	Formatação, Limpeza e organização dos dados**:
Remover e/ou substituir dados não estruturados (tipo NaN = “Não disponível” que dificulta a leitura por um gestor não familiarizado com a linguagem de programação e os jargões da área de dados). Verificar a tipagem dos dados (número com numéro, string com string etc.) e se estão na coluna correta.

**3.	Reestruturação da base e output**:
Uma vez realizadas as etapas 1 e 2 acima, podemos reestruturar a base e criar o arquivo final que será utilizado = planilha Excel. Nesta etapa é importante revisar uma última vez a database em busca de pequenas divergências e erros para correção.



## **Parte 2 — Tratamento e análise**

---

### **a.	Total de solicitações por produto**

---

In [9]:
# Conta o total de solicitações por produto (aprovadas ou não)
total_solicitacoes_por_produto = df['produto'].value_counts()

# Exibir o resultado
print("Total de solicitações por produto:")
print('-' * 35)
display(total_solicitacoes_por_produto)
print('-' * 35)

Total de solicitações por produto:
-----------------------------------


,count
produto,
Emprestimo,343
Cartao,339
Financiamento,338


-----------------------------------


### **b. Taxa de Aprovação por Produto**

---


In [34]:
# Contar o total de solicitações por produto
total_solicitacoes = df['produto'].value_counts().rename('Total Solicitações')

# Contar o número de solicitações aprovadas por produto, usando a coluna 'status' correta
solicitacoes_aprovadas = df[df['status'] == 'Aprovado']['produto'].value_counts().rename('Solicitações Aprovadas')

# Combinar os resultados
taxas_aprovacao_df = pd.concat([total_solicitacoes, solicitacoes_aprovadas], axis=1).fillna(0)

# Calcular a taxa de aprovação
taxas_aprovacao_df['Taxa de Aprovação (%)'] = (taxas_aprovacao_df['Solicitações Aprovadas'] / taxas_aprovacao_df['Total Solicitações'] * 100).round(2)

# Exibir o resultado
print("Taxa de Aprovação por Produto:")
print('-' * 40)
display(taxas_aprovacao_df)

Taxa de Aprovação por Produto:
----------------------------------------


,Total Solicitações,Solicitações Aprovadas,Taxa de Aprovação (%)
produto,,,
Emprestimo,343,119,34.69
Cartao,339,117,34.51
Financiamento,338,116,34.32


**<u>Comentário</u>**: A distribuição de produtos de crédito é bem semelhante, isto é, o número de solicitações por modalidade são próximos entre si. As taxas de aprovação por produto é de aproximadamente 1/3, ou seja, a cada 3 clientes do Agibank que solicita crédito, apenas 1 é aprovado. É necessário mais informações para entender os motivos de recusa. Eu analisei esse ponto com mais detalhes no final do relatório.

### **c.	Total de solicitações por canal**

---

In [10]:
total_solicitacoes_por_canal = df['canal_entrada'].value_counts()

print("Total de solicitações por canal:")
print('-' * 35)
display(total_solicitacoes_por_canal)
print('-' * 35)

Total de solicitações por canal:
-----------------------------------


,count
canal_entrada,
App,362
Web,346
Agencia,312


-----------------------------------


**<u>Comentário</u>**: Podemos ver aqui que mais de 2/3 das solicitações de crédito (69,41%) vem dos meios digitais (App e Web). O meio presencial tem uma participação bem significativa com pouco mais de 30% de participação.

> Minha sugestão aqui é tentar buscar uma informação extra do perfil do cliente, pelo menos a idade dele/dela para verificar se os contratantes via agência são mais velhos. Com isso é possível fazer ações de monitoramento para novas ofertas/renovação de crédito.

### **d. Top 5 cidades com mais solicitações**

In [40]:
top_5_cidades = df['cidade'].value_counts().head(5)

print("Top 5 cidades com mais solicitações:")
print('-' * 40)
display(top_5_cidades)
print('-' * 40)

Top 5 cidades com mais solicitações:
----------------------------------------


,count
cidade,
Sao Paulo,120
Recife,117
Goiania,107
Rio de Janeiro,106
Salvador,105


----------------------------------------


**<u>Comentário</u>**: Regiões SE e NE tem uma base significativa nas solicitações de crédito. Isso pode indicar uma clientela com relacionamento mais aderente com o Agibank.

> Seria interessante implementar ações de marketing digital para ampliar a base ou até mesmo promoções do tipo "cliente que indica novo cliente ganha x% de desconto em um produto ou serviço". São apenas ideias.

### **e. Lista de CPFs com mais de 3 solicitações na base**

In [11]:
cpfs_com_mais_de_3_solicitacoes = df['cpf_cliente'].value_counts()
cpfs_com_mais_de_3_solicitacoes = cpfs_com_mais_de_3_solicitacoes[cpfs_com_mais_de_3_solicitacoes > 3]

print("CPFs com mais de 3 solicitações:")
print('-' * 40)
display(cpfs_com_mais_de_3_solicitacoes)
print('-' * 40)
print(f"Total de clientes com mais de 3 solicitações: {len(cpfs_com_mais_de_3_solicitacoes)}")

CPFs com mais de 3 solicitações:
----------------------------------------


,count
cpf_cliente,


----------------------------------------
Total de clientes com mais de 3 solicitações: 0


**<u>Comentário</u>**: O número de CPFs (clientes) com mais de 3 solicitações de crédito não retornou resultados. Será que a análise está correta ou é algum problema no código. Vou mudar um pouco a abordagem e verificar clientes/CPFs com 2 ou mais solicitações. Só para ter certeza mesmo.

In [12]:
cpfs_com_mais_de_2_solicitacoes = df['cpf_cliente'].value_counts()
cpfs_com_mais_de_2_solicitacoes = cpfs_com_mais_de_2_solicitacoes[cpfs_com_mais_de_2_solicitacoes >= 2]

print("CPFs com mais de 2 solicitações:")
print('-' * 40)
display(cpfs_com_mais_de_2_solicitacoes)
print(f"Total de clientes com 2 ou mais solicitações: {len(cpfs_com_mais_de_2_solicitacoes)}")
print('-' * 40)

CPFs com mais de 2 solicitações:
----------------------------------------


,count
cpf_cliente,
76529899981,2
16414194238,2
92484522636,2
83446206392,2
48812123091,2
88326560670,2
33935823997,2
68148287917,2
32811776648,2


Total de clientes com 2 ou mais solicitações: 20
----------------------------------------


**<u>Comentário</u>**: Perfeito! Embora não temos clientes com 3 ou mais solicitações, as coisas mudam para clientes com pelo menos 2 solicitações de crédito. Minha experiência como bancário na CEF diz que esses clientes são mais propensos a adquirir novos produtos e serviços num futuro próximo e, até mesmo, renovação de limites. É interessante informar a equipe do comercial para monitorar esses clientes para futuros negócios.

Um segundo ponto importante é buscar entender os <u>motivos de recusa/não aprovação de propostas</u>, pois são clientes interessados em manter/estreitar ou criar relacionamento com o Agibank. Seria histórico cadastral? Negativação por órgãos de proteção de crédito como SERASA ou SPC? Falta de produtos de relacionamento mais aderentes? Métricas de avaliação de risco de crédito muito rígidas? São hipóteses que podem ser levantadas neste caso.

Muitas vezes esses clientes tem um impedimento temporário e estão bem propensos a contratar produtos e serviços num futuro próximo. Eu sugereria acompanhamento, monitoramento e abordagem desses clientes em potencial para não perder o negócio para a concorrência.

## **Parte 3 — Automação**

---

Para fazer a automação eu seguiria os seguintes passos:

1. Construção de um **script em Python** que coleta dados do arquivo ou de API (fazendo essa tarefa periodicamente ao longo do dia);
2. Validação, formatação/classificação por tipo e limpeza dos dados, indicando as diretrizes de como eles serão utilizados ao longo do processo.
3. Utilizar ou treinar **LLMs** para gerar relatórios padronizados baseados na base de dados enviada.
4. Acompanhar e verificar se a LLM está realizando as tarefas de acordo com o passo anterior. Corrigir possíveis erros no processo.
5. Criar **documentação** do processo usando repositórios do `github` ou `MkDocs`.

## **Parte 4 — Uso de IA**

---

Essa parte conversa e complementa com a anterior. Ferramentas de IA como ChatGPT, Claude, Copilot, Gemini, Deepseek para citar as mais utilizadas no mercado possuem cada uma características que diferem entre si na forma de pensar e produzir uma resposta de acordo com o prompt que a gente fornece. Sendo assim, eu utilizaria sempre 2 ou 3 IAs para me ajudar nas tarefas diárias para otimizar e automatizar processos.

O objetivo de usar mais de uma é, como disse anteriormente, cada ferramenta possui qualidades de pensamento computacional e aprendizagem de máquina intrínsecas que contribuem de forma diversa para resolver um problema. Em outras palavras, a ideia não seria utilizar a sugestão ou insights fornecidos de uma única IA, mas sim uma **combinação** que complementa o todo.

É claro, cabe ao analista verificar as sugestões, validar se elas estão corretas, uma vez que as IAs costumam alucinar bastante e produzir respostas erradas, implementar, testar e, por fim, aplicar no trabalho do dia a dia.